# 06 — Previsão de Tendência

**Objetivo:** projetar os próximos meses de ocorrências usando regressão linear simples.
**Entrada:** DataFrame de `analise_temporal()` — série mensal indexada por `mes_ano`.
**Saída:** série original + `meses_futuros` linhas com `previsao`, `intervalo_inf`, `intervalo_sup`.
**Próximo passo:** copiar o consolidado para `prever_tendencia()` em `pipeline.py` no Sprint 5.

## Célula 1 — Setup

Rodamos o pipeline até `analise_temporal()` para ter a série mensal pronta.
Além das dependências habituais, importamos `LinearRegression` do scikit-learn
e `scipy.stats.t` para calcular o intervalo de confiança.

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression
from scipy import stats

RAIZ = Path.cwd().parent
sys.path.append(str(RAIZ))

from pipeline import carregar_dados, limpar_dados, gerar_metricas, analise_temporal

ARQUIVO = RAIZ / "data" / "raw" / "BaseDPEvolucaoMensalCisp.csv"

df = gerar_metricas(limpar_dados(carregar_dados(ARQUIVO)))
serie = analise_temporal(df)

print("Entrada (serie_temporal):", serie.shape)
serie.head(3)

## Célula 2 — Entender a entrada

A série está indexada por `mes_ano` (tipo `Period`). Regressão linear precisa de
X numérico — não consegue operar com datas diretamente.

Antes de transformar, vale inspecionar o índice e confirmar que não há lacunas
de meses (meses faltantes quebrariam o sequenciamento numérico).

In [ ]:
print("Tipo do índice:", type(serie.index))
print("Período:", serie.index.min(), "→", serie.index.max())
print("Total de meses:", len(serie))
print()

# Verifica lacunas: diferença entre meses consecutivos deve ser sempre 1
diffs = serie.index[1:].asfreq("M").asi8 - serie.index[:-1].asfreq("M").asi8
lacunas = (diffs != 1).sum()
print(f"Lacunas de meses encontradas: {lacunas}")

serie[["total_ocorrencias"]].describe().round(1)

## Célula 3 — Converter `mes_ano` em variável numérica

A regressão usa X = [0, 1, 2, ..., N-1] — um inteiro sequencial para cada mês.
Esse mapeamento preserva a distância uniforme entre os meses.

`sklearn` espera X com shape `(n_amostras, 1)` — por isso o `reshape(-1, 1)`.
y é o vetor de ocorrências totais que queremos prever.

In [ ]:
n = len(serie)

X = np.arange(n).reshape(-1, 1)   # shape: (n, 1)
y = serie["total_ocorrencias"].values  # shape: (n,)

print("X shape:", X.shape, "  y shape:", y.shape)
print()
print("Primeiros 5 valores de X:", X[:5].ravel())
print("Primeiros 5 valores de y:", y[:5])

## Célula 4 — Treinar o modelo

`fit()` é o treinamento: o modelo encontra os coeficientes `a` (inclinação) e `b`
(intercepto) que minimizam o erro quadrático médio entre a reta e os dados reais.

| atributo | o que representa |
|---|---|
| `coef_[0]` | inclinação — quantas ocorrências por mês em média |
| `intercept_` | valor no mês 0 (ponto de partida da reta) |

In [ ]:
modelo = LinearRegression()
modelo.fit(X, y)

inclinacao = modelo.coef_[0]
direcao    = "subindo" if inclinacao > 0 else "caindo"

print(f"Inclinação (a):  {inclinacao:+.2f} ocorrências/mês  → tendência {direcao}")
print(f"Intercepto  (b): {modelo.intercept_:.1f}")

## Célula 5 — Avaliar a qualidade do modelo (R²)

R² mede quanto da variação nos dados a reta consegue explicar. Vai de 0 a 1.

Para dados de crime, valores acima de 0.6 já são informativos — o fenômeno
tem muito ruído estrutural (sazonalidade, eventos externos) que regressão linear
não captura. Um R² baixo aqui não é falha do código, é propriedade do dado.

In [ ]:
r2 = modelo.score(X, y)

if r2 >= 0.8:
    qualidade = "excelente"
elif r2 >= 0.6:
    qualidade = "informativo para dados de crime"
elif r2 >= 0.3:
    qualidade = "fraco — tendência existe mas com muito ruído"
else:
    qualidade = "muito fraco — regressão linear pode não ser adequada"

print(f"R² = {r2:.4f}  →  {qualidade}")

## Célula 6 — Gerar períodos futuros e prever

`pd.period_range()` cria os `meses_futuros` períodos seguintes ao último da série.
Os índices numéricos continuam a sequência: se a série vai de 0 a N-1, os futuros
vão de N até N + meses_futuros - 1.

`modelo.predict()` aplica a equação `a * x + b` para cada novo x.

In [ ]:
meses_futuros = 3

ultimo_mes   = serie.index.max()
periodos_fut = pd.period_range(start=ultimo_mes + 1, periods=meses_futuros, freq="M")

X_fut    = np.arange(n, n + meses_futuros).reshape(-1, 1)
y_fut    = modelo.predict(X_fut)

print("Meses projetados:", list(periodos_fut))
print("Previsões:       ", y_fut.round(1))

## Célula 7 — Calcular o intervalo de confiança (95%)

O sklearn não calcula intervalo de confiança diretamente. Calculamos manualmente
a partir dos resíduos (diferença entre valor real e valor previsto pelo modelo).

O erro padrão dos resíduos (`s`) quantifica o espalhamento típico dos pontos
em torno da reta. Multiplicamos pelo valor crítico `t` de 95% para obter a
margem de erro — quanto a previsão pode errar para cima ou para baixo.

| variável | significado |
|---|---|
| `residuos` | diferença entre y real e y previsto pela reta |
| `s` | desvio padrão dos resíduos (ruído típico) |
| `t_critico` | multiplicador para 95% de confiança |
| `margem` | `t_critico * s` — largura do intervalo |

In [ ]:
y_pred_treino = modelo.predict(X)
residuos      = y - y_pred_treino

# Graus de liberdade: n observações - 2 parâmetros (a e b)
gl         = n - 2
s          = np.sqrt(np.sum(residuos**2) / gl)
t_critico  = stats.t.ppf(0.975, df=gl)   # bicaudal 95%
margem     = t_critico * s

print(f"Erro padrão dos resíduos (s): {s:.1f}")
print(f"Valor crítico t (95%):        {t_critico:.3f}")
print(f"Margem de erro:               ±{margem:.1f} ocorrências")

## Célula 8 — Montar o DataFrame final

O resultado combina a série original com as linhas futuras. As linhas históricas
recebem `NaN` nas colunas de previsão — elas existem nos dados reais, não no modelo.

Valores de previsão abaixo de zero são forçados a 0 — ocorrências negativas
não existem e são um artefato da extrapolação linear.

In [ ]:
df_futuro = pd.DataFrame(
    {
        "total_ocorrencias": np.nan,
        "taxa_100k_media":   np.nan,
        "media_movel_3":     np.nan,
        "previsao":          np.maximum(y_fut, 0).round(1),
        "intervalo_inf":     np.maximum(y_fut - margem, 0).round(1),
        "intervalo_sup":     (y_fut + margem).round(1),
    },
    index=periodos_fut,
)
df_futuro.index.name = "mes_ano"

resultado = pd.concat([serie, df_futuro])

print("Shape final:", resultado.shape)
print()
resultado.tail(meses_futuros + 2)

## Célula 9 — Validação: testar diferentes filtros

A função vai receber séries filtradas por crime e AISP — o modelo deve
funcionar para qualquer combinação. Testamos quatro casos para garantir
que o R² e as previsões são coerentes em cada um.

Séries curtas (AISP específica) têm menos pontos e tendem a ter R² mais volátil.

In [ ]:
def ajustar_e_avaliar(serie_in, meses=3):
    n_   = len(serie_in)
    X_   = np.arange(n_).reshape(-1, 1)
    y_   = serie_in["total_ocorrencias"].values
    m_   = LinearRegression().fit(X_, y_)
    r2_  = m_.score(X_, y_)
    inc_ = m_.coef_[0]
    prev = m_.predict(np.arange(n_, n_ + meses).reshape(-1, 1))
    return r2_, inc_, prev

casos = [
    ("Todos crimes / todas AISPs", None,          None),
    ("hom_doloso / todas AISPs",   "hom_doloso",  None),
    ("Todos crimes / AISP 1",      None,           "1"),
    ("cvli / AISP 1",              "cvli",         "1"),
]

for label, tc, a in casos:
    s    = analise_temporal(df, tipo_crime=tc, aisp=a)
    r2, inc, prev = ajustar_e_avaliar(s)
    direcao = "↑" if inc > 0 else "↓"
    print(f"{label:35s} | R²={r2:.3f} | tendência {direcao} | próx. 3 meses: {prev.round(0)}")

---
## Consolidado — o que vai para `prever_tendencia()` no pipeline.py

```python
from sklearn.linear_model import LinearRegression
from scipy import stats

def prever_tendencia(
    serie_temporal: pd.DataFrame,
    meses_futuros: int = 3,
) -> pd.DataFrame:
    n = len(serie_temporal)

    X = np.arange(n).reshape(-1, 1)
    y = serie_temporal["total_ocorrencias"].values

    modelo = LinearRegression()
    modelo.fit(X, y)

    r2 = modelo.score(X, y)
    log.info("Regressão linear: R²=%.3f | inclinação=%.2f", r2, modelo.coef_[0])

    residuos  = y - modelo.predict(X)
    s         = np.sqrt(np.sum(residuos**2) / (n - 2))
    t_critico = stats.t.ppf(0.975, df=n - 2)
    margem    = t_critico * s

    ultimo_mes   = serie_temporal.index.max()
    periodos_fut = pd.period_range(start=ultimo_mes + 1, periods=meses_futuros, freq="M")
    X_fut        = np.arange(n, n + meses_futuros).reshape(-1, 1)
    y_fut        = modelo.predict(X_fut)

    df_futuro = pd.DataFrame(
        {
            "total_ocorrencias": np.nan,
            "taxa_100k_media":   np.nan,
            "media_movel_3":     np.nan,
            "previsao":          np.maximum(y_fut, 0).round(1),
            "intervalo_inf":     np.maximum(y_fut - margem, 0).round(1),
            "intervalo_sup":     (y_fut + margem).round(1),
        },
        index=periodos_fut,
    )
    df_futuro.index.name = "mes_ano"

    return pd.concat([serie_temporal, df_futuro])
```